# ft_stemfx — StemFX checkpoint fine-tune on Guitar Rig pairs (Colab)

1. Runtime → Change runtime type → **GPU**
2. Edit the paths in the next cell, then run all.
3. If the session dies: just run everything again — training resumes from `latest_checkpoint.pt` on Drive.

In [ ]:
# ---- paths (edit) -------------------------------------------------------
REPO_URL     = "https://github.com/kuhberaBubo/myFX.git"     # repo that contains ft_stemfx/
STEMFX_URL   = "https://github.com/barry-mir/stemfx.git"
DRIVE_ROOT   = "/content/drive/MyDrive/myFX"
DATASET_ZIP  = f"{DRIVE_ROOT}/dataset.zip"              # zip of the dataset/ folder
RUN_NAME     = "ft_v1"
# -------------------------------------------------------------------------
CODE_DIR  = "/content/myFX"
DATA_ROOT = "/content/dataset/run_20k"                  # local SSD (fast)
CKPT_DIR  = f"{DRIVE_ROOT}/ckpt/{RUN_NAME}"             # on Drive (survives disconnects)
LOG_DIR   = f"{DRIVE_ROOT}/logs/{RUN_NAME}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Private repo: add a GitHub token to Colab **Secrets** (key icon) as `GH_TOKEN`. Public repo: leave it unset.

In [ ]:
import os, shutil, subprocess
try:
    from google.colab import userdata
    tok = userdata.get("GH_TOKEN")
except Exception as e:          # secret missing or "notebook access" off
    tok = None
    print(f"[info] GH_TOKEN not available ({type(e).__name__}) -> cloning without token (public repo only)")
url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL

if os.path.isdir(CODE_DIR) and not os.path.isdir(f"{CODE_DIR}/.git"):
    shutil.rmtree(CODE_DIR)     # leftover from a failed clone
if os.path.isdir(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} pull --ff-only
else:
    r = subprocess.run(["git", "clone", "-q", url, CODE_DIR],
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        raise SystemExit("git clone failed. Private repo -> add a GitHub token as Colab secret "
                         "'GH_TOKEN' (Contents: read) and turn on 'Notebook access'; or make the repo public.")
if not os.path.isfile(f"{CODE_DIR}/stemfx/stemfx/model.py"):
    !git clone -q {STEMFX_URL} {CODE_DIR}/stemfx
!git -C {CODE_DIR} log -1 --oneline

In [ ]:
# stemfx pins numpy<2 -> install without deps to keep Colab's numpy.
!pip install -q -e {CODE_DIR}/stemfx --no-deps
!pip install -q multiafx pyloudnorm huggingface_hub soundfile
import numpy, torch; print("numpy", numpy.__version__, "| torch", torch.__version__)

In [ ]:
# Copy the dataset zip to local disk once per session, then unzip (reading from Drive directly is slow).
import os
if not os.path.isfile(f"{DATA_ROOT}/train.jsonl"):
    !cp "{DATASET_ZIP}" /content/dataset.zip
    !unzip -q -o /content/dataset.zip -d /content && rm /content/dataset.zip
!ls {DATA_ROOT} && wc -l {DATA_ROOT}/train.jsonl {DATA_ROOT}/val.jsonl

### Quick check (~1 min): 64 samples, 1 epoch, separate folder

In [ ]:
!python {CODE_DIR}/ft_stemfx/train.py --config {CODE_DIR}/ft_stemfx/configs/ft_default.yaml     data.root={DATA_ROOT} data.max_train_samples=64 data.max_val_samples=32     train.num_epochs=1 train.resume=null     train.checkpoint_dir=/content/tmp_ckpt train.log_dir=/content/tmp_logs

### Train (resumes automatically)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

In [ ]:
!python {CODE_DIR}/ft_stemfx/train.py --config {CODE_DIR}/ft_stemfx/configs/ft_default.yaml     data.root={DATA_ROOT}     train.checkpoint_dir={CKPT_DIR} train.log_dir={LOG_DIR}

### Evaluate (greedy generation on val)

In [ ]:
!python {CODE_DIR}/ft_stemfx/evaluate.py --ckpt {CKPT_DIR}/best_checkpoint.pt     --root {DATA_ROOT} --max-samples 300 --out {LOG_DIR}/eval_val.json

### Inference
`best_model.pt` (~114MB, model only) is saved next to the checkpoints on Drive. Download it and run `ft_stemfx/infer.py` locally on CPU (no dataset folder needed).